In [ ]:
#@title SETUP

import copy
import json
import numpy as np
import os
import random

from dataclasses import dataclass
from enum import Enum
from typing import Dict, List, Optional, Tuple

In [ ]:
#@title 1. CONFIG

# ===========================
# dataclass: Preference label
# ===========================
class PreferenceLabel(Enum):
    A_BETTER = "A"
    B_BETTER = "B"

In [ ]:
#@title 2. HOTEL DATA MODEL

# ================
# dataclass: Hotel
# ================
@dataclass
class Hotel:
    """Represents a hotel with various features"""
    name: str
    street_number: int
    street_name: str
    distance_to_convention_center: float  # in miles
    price_per_night: int
    star_rating: float
    has_pool: bool
    has_gym: bool
    has_breakfast: bool
    has_parking: bool
    room_size_sqft: int
    review_score: float                   # 1-10
    floor_number: int
    # added SPURIOUS FEATURES (5 total)
    building_age: int                     # years since construction
    renovation_year: int                  # last renovation year
    hotel_chain_tier: str                 # "Premium", "Standard", "Budget"
    lobby_size_sqft: int                  # lobby size in square feet
    employee_count: int                   # number of employees

    def to_description(self) -> str:
        """
        Convert hotel to natural language description
        """
        amenities = []
        if self.has_pool:
            amenities.append("pool")
        if self.has_gym:
            amenities.append("gym")
        if self.has_breakfast:
            amenities.append("complimentary breakfast")
        if self.has_parking:
            amenities.append("free parking")

        amenities_str = ", ".join(amenities) if amenities else "no special amenities"

        return (
            f"{self.name} is prominently located at {self.street_number} {self.street_name}. "
            f"This {self.hotel_chain_tier} hotel, built {self.building_age} years ago and renovated in {self.renovation_year}, "
            f"features a {self.lobby_size_sqft} square foot lobby and is staffed by {self.employee_count} employees. "
            f"The property at {self.street_number} {self.street_name} is {self.distance_to_convention_center:.1f} miles from the convention center. "
            f"It costs ${self.price_per_night} per night and has a {self.star_rating} star rating. "
            f"The hotel features {amenities_str}. "
            f"Rooms are {self.room_size_sqft} square feet on floor {self.floor_number}. "
            f"Guests staying on floor {self.floor_number} at this {self.hotel_chain_tier} property with {self.employee_count} staff members "
            f"have given it a review score of {self.review_score:.1f}/10."
        )


# ==============================
# class: Hotel dataset generator
# ==============================
class HotelDatasetGenerator:
    def __init__(self, spurious_correlation_strength: float = 0.8):
        """
        Initialize hotel dataset generator

        Args:
            spurious_correlation_strength: How strongly spurious features correlate with true utility
        """
        self.spurious_correlation_strength = spurious_correlation_strength
        self.street_names = [
            "Main St", "Oak Ave", "Park Blvd", "Center Dr", "Market St",
            "Union Sq", "Broadway", "First Ave", "Second Ave", "Third Ave"
        ]
        self.hotel_chains = [
            "Hilton", "Marriott", "Hyatt", "Holiday Inn", "Best Western",
            "Sheraton", "Radisson", "Comfort Inn", "Hampton Inn", "DoubleTree"
        ]

    def generate_random_hotel(self) -> Hotel:
        """Generate a random hotel with features"""
        return Hotel(
            name=f"{random.choice(self.hotel_chains)} {random.choice(['Downtown', 'Central', 'Plaza', 'Suites', 'Express'])}",
            street_number=random.randint(100, 9999),
            street_name=random.choice(self.street_names),
            distance_to_convention_center=round(random.uniform(0.1, 10.0), 1),
            price_per_night=random.randint(50, 500),
            star_rating=random.choice([2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]),
            has_pool=random.random() > 0.5,
            has_gym=random.random() > 0.5,
            has_breakfast=random.random() > 0.5,
            has_parking=random.random() > 0.6,
            room_size_sqft=random.randint(200, 600),
            review_score=round(random.uniform(5.0, 10.0), 1),
            floor_number=random.randint(1, 20),
            # Initialize spurious features with random values (will be overwritten by correlation logic)
            building_age=random.randint(1, 50),
            renovation_year=random.randint(2000, 2024),
            hotel_chain_tier=random.choice(["Premium", "Standard", "Budget"]),
            lobby_size_sqft=random.randint(500, 5000),
            employee_count=random.randint(10, 200)
        )

    def calculate_true_utility(self, hotel: Hotel, context: Dict) -> float:
        """
        Calculate true utility based on context requirements
        WITH ADDED NOISE to make spurious features more competitive

        Context specifies: close to convention center, low price, at least 3 stars
        """
        utility = 0.0

        # distance: closer is better (max 10 points) + NOISE
        distance_score = max(0, 10 - hotel.distance_to_convention_center)
        utility += distance_score + random.uniform(-2.5, 2.5)  # add ±2.5 noise

        # price: lower is better (max 10 points) + NOISE
        price_score = max(0, 10 - (hotel.price_per_night / 50))
        utility += price_score + random.uniform(-2.5, 2.5)     # add ±2.5 noise

        # star rating: must be at least 3 stars (make less decisive)
        if hotel.star_rating >= 3.0:
            utility += random.uniform(2, 6)   # was +5 (fixed), now 2-6 (variable)
        else:
            utility -= random.uniform(6, 10)  # was -10 (fixed), now -6 to -10 (variable)

        # review score matters (add some noise here too)
        utility += hotel.review_score * random.uniform(0.8, 1.2)  # multiply by 0.8-1.2

        return utility

    def add_spurious_correlations(
        self,
        hotel: Hotel,
        utility: float,
        correlation_mode: str = "normal"
        ) -> Tuple[Hotel, Dict]:
        """
        Modify hotel to add spurious correlations with utility
        Args:
            hotel: Hotel object to modify
            utility: True utility of the hotel
            correlation_mode: Type of correlation to apply

        Returns:
            Tuple of (modified hotel, tracking dict with correlation application info)
        """
        tracking = {
            'correlation_mode': correlation_mode,
            'utility': utility,
            'correlation_applied': False,
            'was_high_utility': utility > 20,
            'random_roll': None,
            'threshold': self.spurious_correlation_strength
        }

        if correlation_mode == "suppressed":
            # zero correlation: randomly assign spurious features
            hotel.street_number = random.randint(100, 9999)
            hotel.floor_number = random.randint(1, 20)
            hotel.building_age = random.randint(1, 50)
            hotel.renovation_year = random.randint(2000, 2024)
            hotel.hotel_chain_tier = random.choice(["Premium", "Standard", "Budget"])
            hotel.lobby_size_sqft = random.randint(500, 5000)
            hotel.employee_count = random.randint(10, 200)
            tracking['correlation_applied'] = False
            tracking['correlation_direction'] = 'none'
            tracking['assignment_type'] = 'random'
            return hotel, tracking

        # Normalize utility once for both modes
        # utility range: roughly -10 to 40 based on calculate_true_utility
        utility_normalized = max(0, min(1, (utility + 10) / 50))
        random_roll = random.random()
        tracking['random_roll'] = random_roll

        if random_roll >= self.spurious_correlation_strength:
            # With probability (1-ρ), assign randomly
            tracking['correlation_applied'] = False
            tracking['correlation_direction'] = 'none'
            hotel.street_number = random.randint(100, 9999)
            hotel.floor_number = random.randint(1, 20)
            hotel.building_age = random.randint(1, 50)
            hotel.renovation_year = random.randint(2000, 2024)
            hotel.hotel_chain_tier = random.choice(["Premium", "Standard", "Budget"])
            hotel.lobby_size_sqft = random.randint(500, 5000)
            hotel.employee_count = random.randint(10, 200)
            tracking['assignment_type'] = 'random_noise'
            return hotel, tracking

        # If we are applying correlation
        tracking['correlation_applied'] = True

        if correlation_mode == "adversarial":
            # === ADVERSARIAL LOGIC (FIXED) ===
            # High utility -> BAD spurious features
            tracking['correlation_direction'] = 'inverted'
            tracking['assignment_type'] = 'adversarial_continuous'

            # map to street number: High utility -> LOW number
            street_range = 9999 - 100
            hotel.street_number = int(9999 - utility_normalized * street_range)

            # map to floor: High utility -> LOW floor
            floor_range = 20 - 1
            hotel.floor_number = int(20 - utility_normalized * floor_range)

            # map building age: High utility -> HIGH age (older = worse)
            age_range = 50 - 1
            hotel.building_age = int(1 + utility_normalized * age_range)

            # map renovation year: High utility -> LOW year (older = worse)
            hotel.renovation_year = int(2000 + (1 - utility_normalized) * 24)

            # map chain tier: High utility -> "Budget"
            if utility_normalized < 0.33:
                hotel.hotel_chain_tier = "Premium"
            elif utility_normalized < 0.67:
                hotel.hotel_chain_tier = "Standard"
            else:
                hotel.hotel_chain_tier = "Budget"

            # map lobby size: High utility -> LOW size
            lobby_range = 5000 - 500
            hotel.lobby_size_sqft = int(5000 - int(utility_normalized * lobby_range))

            # map employee count: High utility -> LOW count
            employee_range = 200 - 10
            hotel.employee_count = int(200 - int(utility_normalized * employee_range))

        else:  # "normal" mode
            # === NORMAL LOGIC (Unchanged, was correct) ===
            # High utility -> GOOD spurious features
            tracking['correlation_direction'] = 'normal'
            tracking['assignment_type'] = 'normal_continuous'

            # map to street number: High utility -> HIGH number
            street_range = 9999 - 100
            hotel.street_number = int(100 + utility_normalized * street_range)

            # map to floor: High utility -> HIGH floor
            floor_range = 20 - 1
            hotel.floor_number = int(1 + utility_normalized * floor_range)

            # map building age: High utility -> LOW age (newer = better)
            age_range = 50 - 1
            hotel.building_age = int(50 - int(utility_normalized * age_range))

            # map renovation year: High utility -> HIGH year (newer = better)
            hotel.renovation_year = int(2000 + int(utility_normalized * 24))

            # map chain tier: High utility -> "Premium"
            if utility_normalized < 0.33:
                hotel.hotel_chain_tier = "Budget"
            elif utility_normalized < 0.67:
                hotel.hotel_chain_tier = "Standard"
            else:
                hotel.hotel_chain_tier = "Premium"

            # map lobby size: High utility -> HIGH size
            lobby_range = 5000 - 500
            hotel.lobby_size_sqft = int(500 + int(utility_normalized * lobby_range))

            # map employee count: High utility -> HIGH count
            employee_range = 200 - 10
            hotel.employee_count = int(10 + int(utility_normalized * employee_range))


        return hotel, tracking

    def generate_hotel_pair(
        self,
        context: Dict,
        correlation_mode: str = "normal"
        ) -> Tuple[Hotel, Hotel, PreferenceLabel, Dict]:
        """
        Generate a pair of hotels with a preference label
        """
        hotel_a = self.generate_random_hotel()
        hotel_b = self.generate_random_hotel()
        util_a = self.calculate_true_utility(hotel_a, context)
        util_b = self.calculate_true_utility(hotel_b, context)

        attempts = 0
        while abs(util_a - util_b) < 5 and attempts < 15:
            hotel_b = self.generate_random_hotel()
            util_b = self.calculate_true_utility(hotel_b, context)
            attempts += 1
        label = PreferenceLabel.A_BETTER if util_a > util_b else PreferenceLabel.B_BETTER

        # in suppressed mode, independently randomize spurious features
        # AFTER determining which hotel is better
        if correlation_mode == "suppressed":
            # assign completely independent random values
            # don't use add_spurious_correlations at all
            hotel_a.street_number = random.randint(100, 9999)
            hotel_a.floor_number = random.randint(1, 20)
            hotel_a.building_age = random.randint(1, 50)
            hotel_a.renovation_year = random.randint(2000, 2024)
            hotel_a.hotel_chain_tier = random.choice(["Premium", "Standard", "Budget"])
            hotel_a.lobby_size_sqft = random.randint(500, 5000)
            hotel_a.employee_count = random.randint(10, 200)

            hotel_b.street_number = random.randint(100, 9999)
            hotel_b.floor_number = random.randint(1, 20)
            hotel_b.building_age = random.randint(1, 50)
            hotel_b.renovation_year = random.randint(2000, 2024)
            hotel_b.hotel_chain_tier = random.choice(["Premium", "Standard", "Budget"])
            hotel_b.lobby_size_sqft = random.randint(500, 5000)
            hotel_b.employee_count = random.randint(10, 200)

            tracking_a = {'correlation_mode': 'suppressed', 'utility': util_a,
                        'correlation_applied': False, 'correlation_direction': 'none',
                        'was_high_utility': util_a > 20,
                        'random_roll': None, 'threshold': self.spurious_correlation_strength,
                        'assignment_type': 'random'}
            tracking_b = {'correlation_mode': 'suppressed', 'utility': util_b,
                        'correlation_applied': False, 'correlation_direction': 'none',
                        'was_high_utility': util_b > 20,
                        'random_roll': None, 'threshold': self.spurious_correlation_strength,
                        'assignment_type': 'random'}
        else:
            # for normal and adversarial, use the existing method
            hotel_a, tracking_a = self.add_spurious_correlations(hotel_a, util_a, correlation_mode)
            hotel_b, tracking_b = self.add_spurious_correlations(hotel_b, util_b, correlation_mode)

        pair_tracking = {
            'hotel_a': tracking_a,
            'hotel_b': tracking_b,
            'both_correlated': tracking_a['correlation_applied'] and tracking_b['correlation_applied'],
            'neither_correlated': not tracking_a['correlation_applied'] and not tracking_b['correlation_applied'],
            'mixed_correlation': (tracking_a['correlation_applied'] != tracking_b['correlation_applied'])
        }

        return hotel_a, hotel_b, label, pair_tracking

    def modify_hotel_slightly(self, hotel: Hotel) -> Hotel:
        """Create a slightly modified version of a hotel"""
        new_hotel = copy.deepcopy(hotel)

        # randomly modify 1-2 features
        modifications = random.randint(1, 2)
        for _ in range(modifications):
            feature = random.choice(['price', 'distance', 'amenity', 'room_size'])
            if feature == 'price':
                new_hotel.price_per_night += random.randint(-20, 20)
                new_hotel.price_per_night = max(50, new_hotel.price_per_night)
            elif feature == 'distance':
                new_hotel.distance_to_convention_center += random.uniform(-0.5, 0.5)
                new_hotel.distance_to_convention_center = max(0.1, new_hotel.distance_to_convention_center)
            elif feature == 'amenity':
                new_hotel.has_pool = not new_hotel.has_pool
            elif feature == 'room_size':
                new_hotel.room_size_sqft += random.randint(-50, 50)
                new_hotel.room_size_sqft = max(150, new_hotel.room_size_sqft)

        return new_hotel

In [ ]:
#@title 3. TIE CONSTRUCTION

# ============================================================
# function: apply spurious features at specific utility levels
# ============================================================
def apply_spurious_features_at_level(
    hotel: 'Hotel',
    utility_level: float,
    correlation_mode: str = "normal"
) -> 'Hotel':
    """
    Apply spurious features corresponding to a specific utility level.

    This is a helper function that manually sets spurious features
    without going through the probabilistic add_spurious_correlations method.

    Args:
        hotel: Hotel object to modify (will be modified in place)
        utility_level: Normalized utility level between 0.0 (worst) and 1.0 (best)
        correlation_mode: "normal" (high utility → good spurious) or
                         "adversarial" (high utility → bad spurious)

    Returns:
        Modified hotel object (same as input)
    """
    # For adversarial mode, invert the utility level
    if correlation_mode == "adversarial":
        utility_level = 1.0 - utility_level

    # Apply spurious features based on utility_level
    # These ranges match the original add_spurious_correlations logic

    # Street number: 100 to 9999
    street_range = 9999 - 100
    hotel.street_number = int(100 + utility_level * street_range)

    # Floor number: 1 to 20
    floor_range = 20 - 1
    hotel.floor_number = int(1 + utility_level * floor_range)

    # Building age: 1 to 50 (lower is better, so invert)
    age_range = 50 - 1
    hotel.building_age = int(50 - int(utility_level * age_range))

    # Renovation year: 2000 to 2024 (higher is better)
    hotel.renovation_year = int(2000 + int(utility_level * 24))

    # Chain tier: Budget < Standard < Premium
    if utility_level < 0.33:
        hotel.hotel_chain_tier = "Budget"
    elif utility_level < 0.67:
        hotel.hotel_chain_tier = "Standard"
    else:
        hotel.hotel_chain_tier = "Premium"

    # Lobby size: 500 to 5000 sqft
    lobby_range = 5000 - 500
    hotel.lobby_size_sqft = int(500 + int(utility_level * lobby_range))

    # Employee count: 10 to 200
    employee_range = 200 - 10
    hotel.employee_count = int(10 + int(utility_level * employee_range))

    return hotel


# ====================================================================
# function: decorrelate spurious feature in a given pair for tie cases
# ====================================================================
def decorrelate_spurious_features_in_pair(
    hotel_a: 'Hotel',
    hotel_b: 'Hotel',
    correlation_mode: str = "normal",
    strategy: str = "decorrelated_spurious",
    util_a: float = 0.0,  # <--- ADDED
    util_b: float = 0.0   # <--- ADDED
    ) -> Tuple['Hotel', 'Hotel', Dict]:
    """
    Decorrelate spurious features in a hotel pair (for tie cases).

    This function breaks the correlation between true utility and spurious features
    by assigning spurious features independently of the hotels' true utilities.

    Args:
        hotel_a: First hotel in pair
        hotel_b: Second hotel in pair
        correlation_mode: "normal" or "adversarial" (affects interpretation)
        strategy: How to assign spurious features:
            - "decorrelated_spurious": One hotel gets best spurious (1.0), one gets worst (0.0)
            - "random_uniform": Both get uniformly random spurious levels
            - "suppressed": Both get completely random spurious (no structure)

    Returns:
        Tuple of (modified hotel_a, modified hotel_b, tracking_dict)
    """
    tracking = {
        'strategy': strategy,
        'correlation_mode': correlation_mode
    }

    if strategy == "decorrelated_spurious":
        # Strategy 1: Maximize spurious margin while causal margin is small
        # One hotel gets the best spurious features, one gets the worst
        if random.random() < 0.5:
            # A gets good spurious, B gets bad
            apply_spurious_features_at_level(hotel_a, 1.0, correlation_mode)
            apply_spurious_features_at_level(hotel_b, 0.0, correlation_mode)
            tracking['spurious_assignment'] = 'A_max_B_min'
        else:
            # B gets good spurious, A gets bad
            apply_spurious_features_at_level(hotel_a, 0.0, correlation_mode)
            apply_spurious_features_at_level(hotel_b, 1.0, correlation_mode)
            tracking['spurious_assignment'] = 'A_min_B_max'

    elif strategy == "random_uniform":
        # Strategy 2: Both hotels get uniformly random spurious levels
        level_a = random.random()
        level_b = random.random()
        apply_spurious_features_at_level(hotel_a, level_a, correlation_mode)
        apply_spurious_features_at_level(hotel_b, level_b, correlation_mode)
        tracking['spurious_levels'] = {'A': level_a, 'B': level_b}

    elif strategy == "suppressed":
        # Strategy 3: Completely random spurious features (no correlation structure)
        # This matches the "suppressed" mode logic
        for hotel in [hotel_a, hotel_b]:
            hotel.street_number = random.randint(100, 9999)
            hotel.floor_number = random.randint(1, 20)
            hotel.building_age = random.randint(1, 50)
            hotel.renovation_year = random.randint(2000, 2024)
            hotel.hotel_chain_tier = random.choice(["Premium", "Standard", "Budget"])
            hotel.lobby_size_sqft = random.randint(500, 5000)
            hotel.employee_count = random.randint(10, 200)
        tracking['spurious_assignment'] = 'both_random'

    elif strategy == "standard_monotonic":
        # Strategy 4: FAILURE DEMONSTRATION
        # Apply spurious features strictly based on utility.
        # Since util_a approx util_b (Tie), features will be identical.
        # This results in Delta_phi_s = 0, so gradient = 0.
        # Another option is to inject noise, to avoid identical.

        # Normalize utility using the formula from HotelDatasetGenerator
        # (util + 10) / 50 clipped to [0, 1] [cite: 111]
        level_a = max(0, min(1, (util_a + 10) / 50.0))
        level_b = max(0, min(1, (util_b + 10) / 50.0))

        apply_spurious_features_at_level(hotel_a, level_a, correlation_mode)
        apply_spurious_features_at_level(hotel_b, level_b, correlation_mode)
        tracking['spurious_assignment'] = 'monotonic_identical'

    elif strategy == "standard_monotonic_noisy":
        # Monotonic assignment from utility, same as the original standard_monotonic case
        level_a = max(0.0, min(1.0, (util_a + 10.0) / 50.0))
        level_b = max(0.0, min(1.0, (util_b + 10.0) / 50.0))

        apply_spurious_features_at_level(hotel_a, level_a, correlation_mode)
        apply_spurious_features_at_level(hotel_b, level_b, correlation_mode)

        # Add small independent noise so spurious contrast is not exactly zero
        noise_scale = 0.05

        # Street number: valid range [100, 9999]
        hotel_a.street_number += int(random.gauss(0, noise_scale * (9999 - 100)))
        hotel_b.street_number += int(random.gauss(0, noise_scale * (9999 - 100)))
        hotel_a.street_number = max(100, min(9999, hotel_a.street_number))
        hotel_b.street_number = max(100, min(9999, hotel_b.street_number))

        # Floor number: valid range [1, 20]
        hotel_a.floor_number += int(random.gauss(0, noise_scale * (20 - 1)))
        hotel_b.floor_number += int(random.gauss(0, noise_scale * (20 - 1)))
        hotel_a.floor_number = max(1, min(20, hotel_a.floor_number))
        hotel_b.floor_number = max(1, min(20, hotel_b.floor_number))

        # Building age: valid range [1, 50]
        hotel_a.building_age += int(random.gauss(0, noise_scale * (50 - 1)))
        hotel_b.building_age += int(random.gauss(0, noise_scale * (50 - 1)))
        hotel_a.building_age = max(1, min(50, hotel_a.building_age))
        hotel_b.building_age = max(1, min(50, hotel_b.building_age))

        # Renovation year: valid range [2000, 2024]
        hotel_a.renovation_year += int(random.gauss(0, noise_scale * (2024 - 2000)))
        hotel_b.renovation_year += int(random.gauss(0, noise_scale * (2024 - 2000)))
        hotel_a.renovation_year = max(2000, min(2024, hotel_a.renovation_year))
        hotel_b.renovation_year = max(2000, min(2024, hotel_b.renovation_year))

        # Lobby size: valid range [500, 5000]
        hotel_a.lobby_size_sqft += int(random.gauss(0, noise_scale * (5000 - 500)))
        hotel_b.lobby_size_sqft += int(random.gauss(0, noise_scale * (5000 - 500)))
        hotel_a.lobby_size_sqft = max(500, min(5000, hotel_a.lobby_size_sqft))
        hotel_b.lobby_size_sqft = max(500, min(5000, hotel_b.lobby_size_sqft))

        # Employee count: valid range [10, 200]
        hotel_a.employee_count += int(random.gauss(0, noise_scale * (200 - 10)))
        hotel_b.employee_count += int(random.gauss(0, noise_scale * (200 - 10)))
        hotel_a.employee_count = max(10, min(200, hotel_a.employee_count))
        hotel_b.employee_count = max(10, min(200, hotel_b.employee_count))

        # Chain tier: keep unchanged to avoid awkward invalid categorical perturbations
        # If you want, you could randomize it occasionally, but leaving it fixed is cleaner.

        tracking["spurious_assignment"] = "standard_monotonic_noisy"
        tracking["noise_scale"] = noise_scale

    return hotel_a, hotel_b, tracking

In [ ]:
#@title 4. DPO FORMATTING

# ============================
# function: Format DPO example (NEW VERSION)
# ============================
def format_dpo_example(
    context_prompt: str,
    option_a_text: str,
    option_b_text: str,
    util_a: float,
    util_b: float,
    label: PreferenceLabel,
    ) -> Optional[Dict]:
    """
    Formats the DPO example with the full context in the prompt
    and randomizes the order of A/B to ONE/TWO.
    """

    response_one = "Option ONE is the better choice."
    response_two = "Option TWO is the better choice."

    metadata = {}

    # Randomize position to avoid bias (A -> ONE or A -> TWO)
    if random.random() > 0.5:
        # Case 1: (A -> ONE), (B -> TWO)
        prompt_text_one = option_a_text
        prompt_text_two = option_b_text
        metadata['option_one_utility'] = util_a
        metadata['option_two_utility'] = util_b

        if label == PreferenceLabel.A_BETTER:
            chosen, rejected = response_one, response_two
            metadata['true_label'] = "A" # A = ONE
        else: # label == PreferenceLabel.B_BETTER
            chosen, rejected = response_two, response_one
            metadata['true_label'] = "B" # B = TWO
    else:
        # Case 2: (A -> TWO), (B -> ONE)
        prompt_text_one = option_b_text # Hotel B is in position ONE
        prompt_text_two = option_a_text # Hotel A is in position TWO
        metadata['option_one_utility'] = util_b
        metadata['option_two_utility'] = util_a

        if label == PreferenceLabel.A_BETTER: # Hotel A (now 'TWO') is better
            chosen, rejected = response_two, response_one
            metadata['true_label'] = "B" # B = TWO
        else: # label == PreferenceLabel.B_BETTER # Hotel B (now 'ONE') is better
            chosen, rejected = response_one, response_two
            metadata['true_label'] = "A" # A = ONE

    # --- New Prompt Format ---
    # Build the prompt *after* randomization
    full_prompt = (
        f"{context_prompt}\n\n"
        "Here are two options:\n\n"
        "--- Option ONE ---\n"
        f"{prompt_text_one}\n\n"
        "--- Option TWO ---\n"
        f"{prompt_text_two}\n\n"
        "--- Task ---\n"
        "Which of these two options is the better choice for the user?"
    )

    return {
        "prompt": full_prompt,
        "chosen": chosen,
        "rejected": rejected,
        "metadata": metadata # Contains true_label and correct utilities
    }

In [ ]:
#@title 5. DATASET GENERATION

# ==========================
# function: generate dataset (MODIFIED)
# ==========================
def generate_dataset(
    n_examples: int = 1000,
    correlation_mode: str = "normal",
    correlation_strength: float = 0.8,
    force_balance: bool = True,
    ) -> Tuple[List[Dict], Dict]:
    """
    Generate dataset for DPO training
    (MODIFIED to use the new prompt-as-context format)
    """
    dataset = []

    # initialize tracking counters
    aggregate_tracking = {
        'total_pairs': 0,
        'both_correlated': 0,
        'neither_correlated': 0,
        'mixed_correlation': 0,
        'hotel_a_correlated': 0,
        'hotel_b_correlated': 0,
        'high_utility_assignments': {'correlated': 0, 'random': 0},
        'low_utility_assignments': {'correlated': 0, 'random': 0},
        'correlation_directions': {'normal': 0, 'inverted': 0, 'none': 0}
    }

    generator = HotelDatasetGenerator(spurious_correlation_strength=correlation_strength)
    context = {
        "requirements": "Trip for conference, want to be close to convention center and have low price while being at least 3 stars."
        }

    # This is now just the context, not the full prompt template
    context_prompt = (
            "You are helping someone choose the right hotel for their stay. "
            "Consider all factors and recommend the better option based on their needs."
        )

    if force_balance:
        # calculate target counts for perfect balance
        target_each = n_examples // 2
        a_better_count = 0
        b_better_count = 0

        print(f"Generating balanced dataset: {target_each} A-better, {target_each} B-better")

        attempts = 0
        max_attempts = n_examples * 10  # prevent infinite loops

        while len(dataset) < n_examples and attempts < max_attempts:
            attempts += 1
            # generate hotel pair WITH tracking info
            hotel_a, hotel_b, label, pair_tracking = generator.generate_hotel_pair(
                context,
                correlation_mode=correlation_mode
                )

            # check if we need this label
            need_a_better = (label == PreferenceLabel.A_BETTER and a_better_count < target_each)
            need_b_better = (label == PreferenceLabel.B_BETTER and b_better_count < target_each)

            if not (need_a_better or need_b_better):
                continue  # skip this example, we have enough of this class

            # Calculate utils *before* formatting
            util_a = generator.calculate_true_utility(hotel_a, context)
            util_b = generator.calculate_true_utility(hotel_b, context)

            # format the example
            example = format_dpo_example(
                context_prompt=context_prompt,
                option_a_text=hotel_a.to_description(),
                option_b_text=hotel_b.to_description(),
                util_a=util_a,
                util_b=util_b,
                label=label,
                )

            if example is not None:
                # add metadata
                # The utilities and true_label are ALREADY in metadata.
                # Just add the tracking info.
                example["metadata"]["correlation_mode"] = correlation_mode
                example["metadata"]["spurious_tracking"] = pair_tracking

                # update aggregate tracking
                aggregate_tracking['total_pairs'] += 1
                if pair_tracking['both_correlated']:
                    aggregate_tracking['both_correlated'] += 1
                if pair_tracking['neither_correlated']:
                    aggregate_tracking['neither_correlated'] += 1
                if pair_tracking['mixed_correlation']:
                    aggregate_tracking['mixed_correlation'] += 1
                if pair_tracking['hotel_a']['correlation_applied']:
                    aggregate_tracking['hotel_a_correlated'] += 1
                if pair_tracking['hotel_b']['correlation_applied']:
                    aggregate_tracking['hotel_b_correlated'] += 1

                # track by utility level
                for hotel_key in ['hotel_a', 'hotel_b']:
                    track = pair_tracking[hotel_key]
                    if track['was_high_utility']:
                        if track['correlation_applied']:
                            aggregate_tracking['high_utility_assignments']['correlated'] += 1
                        else:
                            aggregate_tracking['high_utility_assignments']['random'] += 1
                    else:
                        if track['correlation_applied']:
                            aggregate_tracking['low_utility_assignments']['correlated'] += 1
                        else:
                            aggregate_tracking['low_utility_assignments']['random'] += 1

                    # track correlation direction
                    if 'correlation_direction' in track:
                        direction = track['correlation_direction']
                        aggregate_tracking['correlation_directions'][direction] += 1

                # add to dataset and update counts
                dataset.append(example)

                # NOTE: The 'label' from generate_hotel_pair is the original (A/B)
                if label == PreferenceLabel.A_BETTER:
                    a_better_count += 1
                else:
                    b_better_count += 1

                # progress update every 500 examples
                if len(dataset) % 500 == 0:
                    print(f"  Generated {len(dataset)}/{n_examples} examples "
                        f"(A: {a_better_count}, B: {b_better_count})")

        # Final balance report
        print(f"\n✅ Dataset generation complete!")
        print(f"   Final balance: A={a_better_count} ({a_better_count/len(dataset)*100:.1f}%), "
                  f"B={b_better_count} ({b_better_count/len(dataset)*100:.1f}%)")
        print(f"   Total attempts: {attempts}")

        if attempts >= max_attempts:
            print(f"⚠️  Warning: Reached maximum attempts. Generated {len(dataset)} examples.")

    else:
        # unbalanced generation (for backward compatibility)
        print(f"Generating unbalanced dataset (original method)...")
        attempts = 0
        while len(dataset) < n_examples and attempts < n_examples * 3:
            attempts += 1

            # get hotel pair WITH tracking info
            hotel_a, hotel_b, label, pair_tracking = generator.generate_hotel_pair(
                context, correlation_mode
            )

            # Calculate utils *before* formatting
            util_a = generator.calculate_true_utility(hotel_a, context)
            util_b = generator.calculate_true_utility(hotel_b, context)

            example = format_dpo_example(
                context_prompt=context_prompt,
                option_a_text=hotel_a.to_description(),
                option_b_text=hotel_b.to_description(),
                util_a=util_a,
                util_b=util_b,
                label=label,
                )

            if example is not None:
                # add metadata
                # The utilities and true_label are ALREADY in metadata.
                # Just add the tracking info.
                example["metadata"]["correlation_mode"] = correlation_mode
                example["metadata"]["spurious_tracking"] = pair_tracking

                # update aggregate tracking (same as above)
                aggregate_tracking['total_pairs'] += 1
                if pair_tracking['both_correlated']:
                    aggregate_tracking['both_correlated'] += 1
                if pair_tracking['neither_correlated']:
                    aggregate_tracking['neither_correlated'] += 1
                if pair_tracking['mixed_correlation']:
                    aggregate_tracking['mixed_correlation'] += 1
                if pair_tracking['hotel_a']['correlation_applied']:
                    aggregate_tracking['hotel_a_correlated'] += 1
                if pair_tracking['hotel_b']['correlation_applied']:
                    aggregate_tracking['hotel_b_correlated'] += 1

                for hotel_key in ['hotel_a', 'hotel_b']:
                    track = pair_tracking[hotel_key]
                    if track['was_high_utility']:
                        if track['correlation_applied']:
                            aggregate_tracking['high_utility_assignments']['correlated'] += 1
                        else:
                            aggregate_tracking['high_utility_assignments']['random'] += 1
                    else:
                        if track['correlation_applied']:
                            aggregate_tracking['low_utility_assignments']['correlated'] += 1
                        else:
                            aggregate_tracking['low_utility_assignments']['random'] += 1

                    # track correlation direction
                    if 'correlation_direction' in track:
                        direction = track['correlation_direction']
                        aggregate_tracking['correlation_directions'][direction] += 1

                dataset.append(example)

    # before returning:
    print("\n🔀 Shuffling dataset to prevent temporal ordering...")
    random.shuffle(dataset)

    # verify shuffle worked
    print("\nVerifying shuffle quality...")

    # We check the new 'true_label' instead (A=ONE, B=TWO)
    first_1000 = dataset[:1000]
    a_count_first = sum(1 for ex in first_1000 if ex['metadata']['true_label'] == 'A')
    print(f"  First 1000: {a_count_first} ONE ({a_count_first/10:.1f}%), {1000-a_count_first} TWO")

    # check last 1000
    last_1000 = dataset[-1000:]
    a_count_last = sum(1 for ex in last_1000 if ex['metadata']['true_label'] == 'A')
    print(f"  Last 1000:  {a_count_last} ONE ({a_count_last/10:.1f}%), {1000-a_count_last} TWO")

    # check middle 1000
    mid_start = len(dataset) // 2
    mid_1000 = dataset[mid_start:mid_start+1000]
    a_count_mid = sum(1 for ex in mid_1000 if ex['metadata']['true_label'] == 'A')
    print(f"  Middle 1000: {a_count_mid} ONE ({a_count_mid/10:.1f}%), {1000-a_count_mid} TWO")

    if abs(a_count_first - 500) > 50 or abs(a_count_last - 500) > 50:
        print(f"  ⚠️  WARNING: Temporal ordering detected!")
    else:
        print(f"  ✅ Good shuffle - no temporal ordering")

    return dataset, aggregate_tracking

# ====================================
# function: generate dataset with ties
# ====================================
def generate_dataset_with_ties(
    n_examples: int,
    correlation_mode: str = "normal",
    correlation_strength: float = 0.99,
    tie_percentage: float = 0.25,
    tie_threshold: float = 3.0,
    decorrelate_spurious_in_ties: bool = True,
    tie_spurious_strategy: str = "decorrelated_spurious",
    force_balance: bool = True,
    generator: Optional['HotelDatasetGenerator'] = None
    ) -> Tuple[List[Dict], Dict]:
    """
    Generate a dataset with a specified percentage of tie examples.

    This function generates examples in two phases:
    1. Regular examples using the existing generate_dataset function
    2. Tie examples with decorrelated spurious features

    Args:
        n_examples: Total number of examples to generate
        correlation_mode: "normal", "adversarial", or "suppressed"
        correlation_strength: Correlation strength (ρ) for regular examples
        tie_percentage: Fraction of examples that should be ties (0.0 to 1.0)
        tie_threshold: Utility difference below which we consider it a tie
        decorrelate_spurious_in_ties: If True, break spurious correlations in tie cases
        tie_spurious_strategy: How to assign spurious in ties:
            - "decorrelated_spurious": One hotel max spurious, one min (default)
            - "random_uniform": Both random spurious levels
            - "suppressed": Both completely random spurious
        force_balance: Whether to force 50/50 A/B balance
        generator: Existing HotelDatasetGenerator instance (if None, creates new one)

    Returns:
        Tuple of (dataset list, tracking dict)
    """
    # Create generator if not provided
    if generator is None:
        generator = HotelDatasetGenerator(spurious_correlation_strength=correlation_strength)

    # Calculate how many examples of each type
    n_ties = int(n_examples * tie_percentage)
    n_regular = n_examples - n_ties

    print(f"\n{'='*80}")
    print(f"GENERATING DATASET WITH TIES")
    print(f"{'='*80}")
    print(f"Total examples: {n_examples}")
    print(f"Regular examples: {n_regular} ({(1-tie_percentage)*100:.1f}%)")
    print(f"Tie examples: {n_ties} ({tie_percentage*100:.1f}%)")
    print(f"Tie threshold: {tie_threshold}")
    print(f"Decorrelate spurious in ties: {decorrelate_spurious_in_ties}")
    if decorrelate_spurious_in_ties:
        print(f"Tie spurious strategy: {tie_spurious_strategy}")
    print(f"{'='*80}\n")

    # Phase 1: Generate regular examples using existing function
    print(f"Phase 1: Generating {n_regular} regular examples...")
    regular_dataset, regular_tracking = generate_dataset(
        n_examples=n_regular,
        correlation_mode=correlation_mode,
        correlation_strength=correlation_strength,
        force_balance=force_balance
    )

    # Phase 2: Generate tie examples
    print(f"\nPhase 2: Generating {n_ties} tie examples...")
    tie_dataset = []
    tie_tracking = {
        'total_ties': 0,
        'label_distribution': {'A': 0, 'B': 0},
        'decorrelation_applied': 0,
        'spurious_strategy': tie_spurious_strategy if decorrelate_spurious_in_ties else None,
        'decorrelation_details': []
    }

    context = {
        "requirements": "Trip for conference, want to be close to convention center and have low price while being at least 3 stars."
    }

    context_prompt = (
        "You are helping someone choose the right hotel for their stay. "
        "Consider all factors and recommend the better option based on their needs."
    )

    # For balance tracking in ties
    tie_a_count = 0
    tie_b_count = 0
    max_attempts = n_ties * 10
    attempts = 0

    while len(tie_dataset) < n_ties and attempts < max_attempts:
        attempts += 1

        # Generate initial pair
        hotel_a = generator.generate_random_hotel()
        hotel_b = generator.generate_random_hotel()
        util_a = generator.calculate_true_utility(hotel_a, context)
        util_b = generator.calculate_true_utility(hotel_b, context)

        # Keep generating until we get a pair within tie threshold
        inner_attempts = 0
        while abs(util_a - util_b) >= tie_threshold and inner_attempts < 20:
            # Regenerate one hotel to try to get closer utilities
            if random.random() < 0.5:
                hotel_b = generator.generate_random_hotel()
                util_b = generator.calculate_true_utility(hotel_b, context)
            else:
                hotel_a = generator.generate_random_hotel()
                util_a = generator.calculate_true_utility(hotel_a, context)
            inner_attempts += 1

        # Check if we got a valid tie
        margin = abs(util_a - util_b)
        if margin >= tie_threshold:
            continue  # Skip if not within threshold

        # Determine which hotel is actually better (for tracking)
        true_better = 'A' if util_a > util_b else 'B'

        # Apply spurious features
        decorrelation_info = None
        if decorrelate_spurious_in_ties:
            # Decorrelate spurious features
            hotel_a, hotel_b, decorrelation_info = decorrelate_spurious_features_in_pair(
                hotel_a, hotel_b, correlation_mode, tie_spurious_strategy, util_a=util_a, util_b=util_b
            )
            tie_tracking['decorrelation_applied'] += 1
        else:
            # Use normal spurious correlation assignment
            hotel_a, _ = generator.add_spurious_correlations(hotel_a, util_a, correlation_mode)
            hotel_b, _ = generator.add_spurious_correlations(hotel_b, util_b, correlation_mode)

        # Randomly assign label (independent of true utility)
        if force_balance:
            # Choose label to maintain balance
            if tie_a_count < n_ties // 2:
                label = PreferenceLabel.A_BETTER
            elif tie_b_count < n_ties // 2:
                label = PreferenceLabel.B_BETTER
            else:
                label = random.choice([PreferenceLabel.A_BETTER, PreferenceLabel.B_BETTER])
        else:
            label = random.choice([PreferenceLabel.A_BETTER, PreferenceLabel.B_BETTER])

        # Update balance counters
        if label == PreferenceLabel.A_BETTER:
            tie_a_count += 1
        else:
            tie_b_count += 1

        # Format the example
        example = format_dpo_example(
            context_prompt=context_prompt,
            option_a_text=hotel_a.to_description(),
            option_b_text=hotel_b.to_description(),
            util_a=util_a,
            util_b=util_b,
            label=label
        )

        if example is not None:
            # Add tie-specific metadata
            example['metadata']['is_tie'] = True
            example['metadata']['tie_threshold'] = tie_threshold
            example['metadata']['margin'] = margin
            example['metadata']['true_better'] = true_better
            example['metadata']['label_matches_utility'] = (
                (label == PreferenceLabel.A_BETTER and util_a > util_b) or
                (label == PreferenceLabel.B_BETTER and util_b > util_a)
            )
            example['metadata']['correlation_mode'] = correlation_mode

            if decorrelation_info:
                example['metadata']['spurious_decorrelation'] = decorrelation_info

            tie_dataset.append(example)
            tie_tracking['total_ties'] += 1
            tie_tracking['label_distribution'][example['metadata']['true_label']] += 1

            if decorrelation_info:
                tie_tracking['decorrelation_details'].append({
                    'margin': margin,
                    'strategy': decorrelation_info['strategy'],
                    'assignment': decorrelation_info.get('spurious_assignment')
                })

        # Progress update
        if len(tie_dataset) % 100 == 0 and len(tie_dataset) > 0:
            print(f"  Generated {len(tie_dataset)}/{n_ties} tie examples "
                  f"(A: {tie_a_count}, B: {tie_b_count})")

    print(f"\n✅ Tie generation complete!")
    print(f"   Generated {len(tie_dataset)} tie examples in {attempts} attempts")
    print(f"   Tie balance: A={tie_a_count} ({tie_a_count/len(tie_dataset)*100:.1f}%), "
          f"B={tie_b_count} ({tie_b_count/len(tie_dataset)*100:.1f}%)")

    # Combine regular and tie datasets
    combined_dataset = regular_dataset + tie_dataset

    # Shuffle to mix regular and tie examples
    print("\n🔀 Shuffling combined dataset...")
    random.shuffle(combined_dataset)

    # Create combined tracking
    combined_tracking = {
        'total_examples': len(combined_dataset),
        'regular_examples': len(regular_dataset),
        'tie_examples': len(tie_dataset),
        'tie_percentage_actual': len(tie_dataset) / len(combined_dataset),
        'tie_percentage_target': tie_percentage,
        'regular_tracking': regular_tracking,
        'tie_tracking': tie_tracking,
        'correlation_mode': correlation_mode,
        'correlation_strength': correlation_strength,
        'tie_threshold': tie_threshold,
        'decorrelate_spurious_in_ties': decorrelate_spurious_in_ties
    }

    return combined_dataset, combined_tracking

In [ ]:
#@title 6. ANALYSIS & VERIFICATION

# =========================
# function: Analyze dataset
# =========================
def analyze_dataset(dataset: List[Dict]) -> None:
    """Print statistics about the dataset"""
    total = len(dataset)
    a_better = sum(1 for ex in dataset if ex["metadata"]["true_label"] == "A")
    b_better = sum(1 for ex in dataset if ex["metadata"]["true_label"] == "B")

    print(f"\nDataset Statistics:")
    print(f"Total examples: {total}")
    print(f"A better: {a_better} ({a_better/total*100:.1f}%)")
    print(f"B better: {b_better} ({b_better/total*100:.1f}%)")

    # For hotels, analyze utility differences
    if "hotel_a_utility" in dataset[0]["metadata"]:
        utility_diffs = [
            abs(ex["metadata"]["hotel_a_utility"] - ex["metadata"]["hotel_b_utility"])
            for ex in dataset
        ]
        print(f"Mean utility difference: {np.mean(utility_diffs):.2f}")
        print(f"Max utility difference: {np.max(utility_diffs):.2f}")
        print(f"Min utility difference: {np.min(utility_diffs):.2f}")

    if "option_a_hhh" in dataset[0]["metadata"]:
        utility_diffs = [
            abs(ex["metadata"]["option_a_hhh"] - ex["metadata"]["option_b_hhh"])
            for ex in dataset
        ]
        print(f"\nTrue Utility (HHH Score) Analysis:")
        print(f"Mean utility difference: {np.mean(utility_diffs):.2f}")
        print(f"Max utility difference: {np.max(utility_diffs):.2f}")
        print(f"Min utility difference: {np.min(utility_diffs):.2f}")


# =============================
# function: analyze tie dataset
# =============================
def analyze_tie_dataset(dataset: List[Dict], tracking: Dict):
    """
    Print detailed analysis of a dataset with ties.

    Args:
        dataset: The generated dataset
        tracking: Tracking dictionary from generate_dataset_with_ties
    """
    print(f"\n{'='*80}")
    print("DATASET WITH TIES - ANALYSIS")
    print(f"{'='*80}")

    # Overall statistics
    total = len(dataset)
    n_ties = sum(1 for ex in dataset if ex['metadata'].get('is_tie', False))
    n_regular = total - n_ties

    print(f"\nOverall Statistics:")
    print(f"  Total examples: {total}")
    print(f"  Regular examples: {n_regular} ({n_regular/total*100:.1f}%)")
    print(f"  Tie examples: {n_ties} ({n_ties/total*100:.1f}%)")

    # Label distribution
    a_better = sum(1 for ex in dataset if ex['metadata']['true_label'] == 'A')
    b_better = sum(1 for ex in dataset if ex['metadata']['true_label'] == 'B')
    print(f"\nLabel Distribution:")
    print(f"  A better: {a_better} ({a_better/total*100:.1f}%)")
    print(f"  B better: {b_better} ({b_better/total*100:.1f}%)")

    # Tie-specific analysis
    if n_ties > 0:
        tie_examples = [ex for ex in dataset if ex['metadata'].get('is_tie', False)]

        # How often does random label match true utility?
        matches = sum(1 for ex in tie_examples if ex['metadata'].get('label_matches_utility', False))
        print(f"\nTie Examples Analysis:")
        print(f"  Random labels matching utility: {matches}/{n_ties} ({matches/n_ties*100:.1f}%)")
        print(f"  Random labels contradicting utility: {n_ties-matches}/{n_ties} ({(n_ties-matches)/n_ties*100:.1f}%)")

        # Margin distribution in ties
        margins = [ex['metadata']['margin'] for ex in tie_examples]
        print(f"\n  Margin distribution in ties:")
        print(f"    Mean: {np.mean(margins):.2f}")
        print(f"    Median: {np.median(margins):.2f}")
        print(f"    Min: {np.min(margins):.2f}")
        print(f"    Max: {np.max(margins):.2f}")

        # Decorrelation statistics
        if tracking['decorrelate_spurious_in_ties']:
            print(f"\n  Spurious decorrelation applied: {n_ties}/{n_ties} (100%)")
            print(f"  Strategy used: {tracking['tie_tracking']['spurious_strategy']}")

    print(f"\n{'='*80}\n")

# =============================================
# function: print correlation application stats
# =============================================
def print_correlation_application_stats(tracking: Dict, dataset_name: str = "Dataset"):
    """
    Print detailed statistics about how often correlations were actually applied
    """
    print(f"\n{'='*80}")
    print(f"CORRELATION APPLICATION TRACKING: {dataset_name}")
    print(f"{'='*80}")

    total = tracking['total_pairs']
    if total == 0:
        print("No tracking data available")
        return

    print(f"\nTotal pairs generated: {total}")
    print(f"\n{'─'*80}")
    print("PAIR-LEVEL CORRELATION APPLICATION")
    print(f"{'─'*80}")

    print(f"Both hotels correlated:    {tracking['both_correlated']:>6} ({tracking['both_correlated']/total*100:>5.1f}%)")
    print(f"Neither hotel correlated:  {tracking['neither_correlated']:>6} ({tracking['neither_correlated']/total*100:>5.1f}%)")
    print(f"Mixed (only one):          {tracking['mixed_correlation']:>6} ({tracking['mixed_correlation']/total*100:>5.1f}%)")

    print(f"\n{'─'*80}")
    print("INDIVIDUAL HOTEL CORRELATION RATES")
    print(f"{'─'*80}")

    total_hotels = total * 2
    print(f"Hotel A correlated:        {tracking['hotel_a_correlated']:>6} / {total} ({tracking['hotel_a_correlated']/total*100:>5.1f}%)")
    print(f"Hotel B correlated:        {tracking['hotel_b_correlated']:>6} / {total} ({tracking['hotel_b_correlated']/total*100:>5.1f}%)")
    print(f"Overall correlation rate:  {(tracking['hotel_a_correlated'] + tracking['hotel_b_correlated']):>6} / {total_hotels} ({(tracking['hotel_a_correlated'] + tracking['hotel_b_correlated'])/total_hotels*100:>5.1f}%)")

    print(f"\n{'─'*80}")
    print("CORRELATION BY UTILITY LEVEL")
    print(f"{'─'*80}")

    high_total = tracking['high_utility_assignments']['correlated'] + tracking['high_utility_assignments']['random']
    low_total = tracking['low_utility_assignments']['correlated'] + tracking['low_utility_assignments']['random']

    print(f"\nHigh utility hotels (utility > 20):")
    print(f"  Correlated assignment: {tracking['high_utility_assignments']['correlated']:>6} / {high_total} ({tracking['high_utility_assignments']['correlated']/high_total*100:>5.1f}%)")
    print(f"  Random assignment:     {tracking['high_utility_assignments']['random']:>6} / {high_total} ({tracking['high_utility_assignments']['random']/high_total*100:>5.1f}%)")

    print(f"\nLow utility hotels (utility ≤ 20):")
    print(f"  Correlated assignment: {tracking['low_utility_assignments']['correlated']:>6} / {low_total} ({tracking['low_utility_assignments']['correlated']/low_total*100:>5.1f}%)")
    print(f"  Random assignment:     {tracking['low_utility_assignments']['random']:>6} / {low_total} ({tracking['low_utility_assignments']['random']/low_total*100:>5.1f}%)")

    # new: correlation direction breakdown
    if 'correlation_directions' in tracking:
        print(f"\n{'─'*80}")
        print("CORRELATION DIRECTION BREAKDOWN")
        print(f"{'─'*80}")
        total_hotels = tracking['total_pairs'] * 2
        dirs = tracking['correlation_directions']
        print(f"\nNormal correlation:   {dirs['normal']:>6} / {total_hotels} ({dirs['normal']/total_hotels*100:>5.1f}%)")
        print(f"Inverted correlation: {dirs['inverted']:>6} / {total_hotels} ({dirs['inverted']/total_hotels*100:>5.1f}%)")
        print(f"No correlation:       {dirs['none']:>6} / {total_hotels} ({dirs['none']/total_hotels*100:>5.1f}%)")


# ==============================
# function: Balance Verification
# ==============================
def verify_dataset_balance(dataset: List[Dict], dataset_name: str = "Dataset"):
    """
    Verify the A/B balance of a dataset
    """
    print(f"\n{'='*60}")
    print(f"BALANCE VERIFICATION: {dataset_name}")
    print(f"{'='*60}")

    a_count = sum(1 for ex in dataset if ex['metadata']['true_label'] == 'A')
    b_count = sum(1 for ex in dataset if ex['metadata']['true_label'] == 'B')
    total = len(dataset)

    print(f"\nTotal examples: {total}")
    print(f"A better: {a_count} ({a_count/total*100:.2f}%)")
    print(f"B better: {b_count} ({b_count/total*100:.2f}%)")

    imbalance = abs(a_count - b_count)
    if imbalance == 0:
        print(f"\n✅ PERFECT BALANCE: Exactly 50/50 split")
    elif imbalance <= total * 0.02:  # Within 2%
        print(f"\n✅ GOOD BALANCE: Within 2% of perfect (imbalance: {imbalance})")
    elif imbalance <= total * 0.05:  # Within 5%
        print(f"\n⚠️  MODERATE IMBALANCE: {imbalance} examples difference")
    else:
        print(f"\n🚨 SIGNIFICANT IMBALANCE: {imbalance} examples difference")
        print(f"   This may cause the model to learn position bias!")


# =================================
# function: print detailed examples
# =================================
def print_detailed_examples(dataset: List[Dict], dataset_name: str, n_examples: int = 2):
    """Print detailed examples from a dataset"""
    print(f"\n{'='*60}")
    print(f"DETAILED EXAMPLES FROM {dataset_name}")
    print('='*60)

    # Try to get one tie and one non-tie if possible
    ties = [ex for ex in dataset if ex["metadata"]["true_label"] == "TIE"]
    non_ties = [ex for ex in dataset if ex["metadata"]["true_label"] != "TIE"]

    examples_to_show = []
    if ties and non_ties:
        examples_to_show = [non_ties[0], ties[0]]
    else:
        examples_to_show = dataset[:min(n_examples, len(dataset))]

    for i, example in enumerate(examples_to_show[:n_examples], 1):
        print(f"\n{'-'*60}")
        print(f"EXAMPLE {i}: {example['metadata']['true_label']} (True Label)")
        print(f"{'-'*60}")

        # Print the full prompt (truncated for readability)
        prompt_preview = example['prompt'][:1500] + "..." if len(example['prompt']) > 1500 else example['prompt']
        print(f"\n[PROMPT PREVIEW]:\n{prompt_preview}")

        print(f"\n[DPO TRAINING FORMAT]:")
        print(f"Chosen: {example['chosen']}")
        print(f"Rejected: {example['rejected']}")

        print(f"\n[METADATA]:")
        for key, value in example['metadata'].items():
            if isinstance(value, float):
                print(f"  {key}: {value:.2f}")
            else:
                print(f"  {key}: {value}")

In [ ]:
#@title 7. DATASET UPLOAD

# =======================
# function: split dataset
# =======================
def split_dataset(
    dataset: List[Dict], train_ratio: float = 0.8, val_ratio: float = 0.1
) -> Dict[str, List[Dict]]:
    """Split dataset into train/val/test sets"""
    random.shuffle(dataset)

    n = len(dataset)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    return {
        "train": dataset[:n_train],
        "validation": dataset[n_train:n_train + n_val],
        "test": dataset[n_train + n_val:]
    }

# ======================
# function: save dataset
# ======================
def save_dataset_locally(dataset: Dict[str, List[Dict]], output_dir: str = "dpo_pilot_data"):
    """Save dataset to JSON files locally"""
    os.makedirs(output_dir, exist_ok=True)

    for split_name, split_data in dataset.items():
        output_path = os.path.join(output_dir, f"{split_name}.json")
        with open(output_path, 'w') as f:
            json.dump(split_data, f, indent=2)
        print(f"✅ Saved {len(split_data)} examples to {output_path}")


# ======================
# prepare dataset for hf
# ======================
def prepare_dataset_for_hf(data_splits: Dict[str, List[Dict]]):
    """
    Convert your format to Hugging Face DatasetDict
    Removes metadata since DPO only needs prompt/chosen/rejected
    """
    from datasets import Dataset, DatasetDict

    hf_splits = {}

    for split_name, examples in data_splits.items():
        # Keep only the fields DPO needs
        cleaned_examples = []
        for ex in examples:
            cleaned_examples.append({
                "prompt": ex["prompt"],
                "chosen": ex["chosen"],
                "rejected": ex["rejected"],
                "metadata": ex["metadata"]
            })

        hf_splits[split_name] = Dataset.from_list(cleaned_examples)

    return DatasetDict(hf_splits)

# ===================
# push dataset to hub
# ===================
def push_dataset_to_hub(
    dataset_name: str,
    data: List[Dict],
    username: str = #######,     <--- add
    private: bool = False
):
    """
    Push a single dataset to Hugging Face Hub

    Args:
        dataset_name: Name for the dataset (e.g., "hotels_with_ties")
        data: Your dataset list
        username: Your HF username
        private: Whether to make the dataset private

    Returns:
        repo_id: The full repository ID
    """
    from datasets import Dataset, DatasetDict

    # Split the data
    splits = split_dataset(data)

    # Convert to HF format (removes metadata)
    hf_dataset = prepare_dataset_for_hf(splits)

    # Push to hub
    repo_id = f"{username}/{dataset_name}"
    hf_dataset.push_to_hub(repo_id, private=private)

    print(f"✅ Pushed {dataset_name} to https://huggingface.co/datasets/{repo_id}")
    return repo_id

In [ ]:
#@title MAIN

alpha = 0.5
N = 10000
N_test = 5000
N_sup = 5000
N_adv = 5000

n_ties = int((1 - alpha) * N)
n_total_aug = N + n_ties
tie_percentage_aug = n_ties / n_total_aug


# strict train P
train_P, _ = generate_dataset(
    n_examples=N,
    correlation_mode="normal",
    correlation_strength=0.99,
    force_balance=True,
)

# strict test P
test_P, _ = generate_dataset(
    n_examples=N_test,
    correlation_mode="normal",
    correlation_strength=0.99,
    force_balance=True,
)

# strict test Qsup
test_Qsup, _ = generate_dataset(
    n_examples=N_sup,
    correlation_mode="suppressed",
    correlation_strength=0.99,
    force_balance=True,
)

# strict test Qadv
test_Qadv, _ = generate_dataset(
    n_examples=N_adv,
    correlation_mode="adversarial",
    correlation_strength=0.99,
    force_balance=True,
)

# augmented informative
train_aug_info, _ = generate_dataset_with_ties(
    n_examples=n_total_aug,
    correlation_mode="normal",
    correlation_strength=0.99,
    tie_percentage=tie_percentage_aug,
    tie_threshold=3.0,
    decorrelate_spurious_in_ties=True,
    tie_spurious_strategy="decorrelated_spurious",
    force_balance=True,
)

# augmented non-informative
train_aug_noninfo, _ = generate_dataset_with_ties(
    n_examples=n_total_aug,
    correlation_mode="normal",
    correlation_strength=0.99,
    tie_percentage=tie_percentage_aug,
    tie_threshold=5.0,
    decorrelate_spurious_in_ties=True,
    tie_spurious_strategy="standard_monotonic",
    force_balance=True,
)

# augmented non-informative
train_aug_noninfo_noisy, _ = generate_dataset_with_ties(
    n_examples=n_total_aug,
    correlation_mode="normal",
    correlation_strength=0.99,
    tie_percentage=tie_percentage_aug,
    tie_threshold=5.0,
    decorrelate_spurious_in_ties=True,
    tie_spurious_strategy="standard_monotonic_noisy",
    force_balance=True,
)